# 08 · Text-to-Image

> **Source notes:** `TextToImage.md`

Prompt engineering, img2img, and spatial control — from text to pictures.

This notebook:
- Implements **img2img** on our MNIST CondUNet (partial noise + denoise)
- Visualises how **strength** controls how much of the source image is preserved
- Demonstrates **negative conditioning** (the negative-prompt extension of CFG)
- Simulates a **ControlNet**-style structural injection (edge map as extra input)
- PixelSmith v5 bonus: real ControlNet via `diffusers` (commented, optional)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Call `run()` to produce the result
#
# Hint:
#    # implement using the APIs described above

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "torch", "torchvision", "matplotlib", "numpy", "-q"], check=True)
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
import torchvision.transforms as T
import numpy as np, matplotlib.pyplot as plt
from torch.utils.data import DataLoader
print("Ready.")

## 1 · Rebuild Conditional U-Net + Train (from Ch.5/6)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `T_STEPS` using `sqrt()`
# 2. Process data
# 3. Define helper function
# 4. Compute `model` using `CondUNet()`
# 5. Call `view()` to produce the result
#
# Hint:
#    enc1 = ResBlock(???)
#    enc2 = ResBlock(???)
#    betas = torch.linspace(???)
#    alpha_bar = torch.cumprod(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
T_STEPS = 1000
betas     = torch.linspace(1e-4, 0.02, T_STEPS)
alphas    = 1 - betas
alpha_bar = torch.cumprod(alphas, 0)
sqrt_ab   = alpha_bar.sqrt()
sqrt_1mab = (1 - alpha_bar).sqrt()

NUM_CLASSES, NULL_CLASS = 10, 10

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half  = self.dim // 2
        freqs = torch.exp(-torch.arange(half, device=t.device) * (np.log(10000)/(half-1)))
        args  = t[:,None].float() * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=-1)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1  = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2  = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.t_proj = nn.Linear(time_dim, out_ch)
        self.skip   = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t_emb):
        h = F.silu(self.conv1(x))
        h = h + self.t_proj(t_emb)[:, :, None, None]
        return F.silu(self.conv2(h)) + self.skip(x)

class CondUNet(nn.Module):
    def __init__(self, base_ch=32, time_dim=64):
        super().__init__()
        self.class_embed = nn.Embedding(NUM_CLASSES + 1, time_dim)
        self.time_embed  = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim*4), nn.SiLU(),
            nn.Linear(time_dim*4, time_dim))
        C = base_ch
        self.enc1 = ResBlock(1, C,   time_dim)
        self.enc2 = ResBlock(C, C*2, time_dim)
        self.enc3 = ResBlock(C*2, C*4, time_dim)
        self.bot  = ResBlock(C*4, C*4, time_dim)
        self.dec3 = ResBlock(C*8, C*2, time_dim)
        self.dec2 = ResBlock(C*4, C,   time_dim)
        self.dec1 = ResBlock(C*2, C,   time_dim)
        self.out  = nn.Conv2d(C, 1, 1)
        self.down = nn.MaxPool2d(2)
        self.up   = nn.Upsample(scale_factor=2, mode="nearest")
    def forward(self, x, t, c):
        te = self.time_embed(t) + self.class_embed(c)
        s1 = self.enc1(x, te); s2 = self.enc2(self.down(s1), te)
        s3 = self.enc3(self.down(s2), te); b = self.bot(s3, te)
        d3 = self.dec3(torch.cat([self.up(b), s3], 1), te)
        d2 = self.dec2(torch.cat([self.up(d3), s2], 1), te)
        d1 = self.dec1(torch.cat([d2, s1], 1), te)
        return self.out(d1)

model  = CondUNet()
optim  = torch.optim.Adam(model.parameters(), lr=2e-4)
dataset = torchvision.datasets.MNIST(
    "/tmp/mnist", train=True, download=True,
    transform=T.Compose([T.ToTensor(), T.Lambda(lambda x: x*2-1)]))
loader = DataLoader(dataset, batch_size=256, shuffle=True)

model.train()
for epoch in range(20):
    for x0, c in loader:
        t = torch.randint(0, T_STEPS, (len(x0),))
        eps = torch.randn_like(x0)
        xt  = sqrt_ab[t].view(-1,1,1,1)*x0 + sqrt_1mab[t].view(-1,1,1,1)*eps
        mask = torch.rand(len(x0)) < 0.10
        c_in = c.clone(); c_in[mask] = NULL_CLASS
        loss = F.mse_loss(model(xt, t, c_in), eps)
        optim.zero_grad(); loss.backward(); optim.step()
    if (epoch+1) % 5 == 0: print(f"Epoch {epoch+1}/20  loss={loss.item():.4f}")
print("Training done.")

## 2 · img2img — Strength Sweep

Start from a real MNIST digit but ask the model to re-draw it as a different class.

In [ ]:
def img2img_ddim(model, x_src, target_class, strength=0.7, n_steps=30, w=3.0):
    """
    TODO #3: Implement `img2img_ddim()`.

    Steps:
    1. Call `DataLoader()` to produce the result
    2. Define helper function `img2img_ddim()`
    3. Call `randn_like()` to produce the result
    4. Call `model()` to produce the result
    5. Compute `strengths`
    6. Compute `results`
    7. Plot results -- call `imshow()`
    8. Plot results -- call `suptitle()`

    Hint:
    cls = torch.full(???)
    null = torch.full(???)
    eps_fwd = torch.randn_like(???)
    ts = torch.linspace(???)

    Returns: x
    """
    raise NotImplementedError("TODO: implement img2img_ddim()")

## 3 · Negative Conditioning

Replace the null-token unconditional with a *negative class* baseline. This pushes the output away from an unwanted class.

In [ ]:
def cfg_negative_sample(model, target_class, negative_class, w=5.0, n=4, n_steps=40):
    """
    TODO #4: Implement `cfg_negative_sample()`.

    Steps:
    1. Define helper function `cfg_negative_sample()`
    2. Compute `target` using `negative()`
    3. Plot results -- call `subplots()`
    4. Plot results -- call `suptitle()`

    Hint:
    pos = torch.full(???)
    neg = torch.full(???)
    x = torch.randn(???)
    ts = torch.linspace(???)

    Returns: x
    """
    raise NotImplementedError("TODO: implement cfg_negative_sample()")

## 4 · Mini ControlNet Simulation

Add an **edge map** (Sobel filter output of source image) as an extra input channel, training the model to match both the edge structure and the class label.

In [ ]:
def sobel_edges(x):
    """
    TODO #5: Implement `sobel_edges()`.

    Steps:
    1. Process data
    2. Define helper function `sobel_edges()`
    3. Process data
    4. Compute `edges`
    5. Plot results -- call `imshow()`

    Hint:
    kx = torch.tensor(???)
    ky = kx.transpose(???)
    gx = F.conv2d(???)
    gy = F.conv2d(???)

    Returns: (gx.pow(2) + gy.pow(2)).sqrt().clamp(...
    """
    raise NotImplementedError("TODO: implement sobel_edges()")

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Define helper function
# 2. Compute `ctrl_model` using `EdgeCondUNet()`
# 3. Call `view()` to produce the result
#
# Hint:
#    enc1 = ResBlock(???)
#    enc2 = ResBlock(???)
#    class_embed = nn.Embedding(???)
#    time_embed = nn.Sequential(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# Mini ControlNet: accept 2-channel input (noisy image + edge map)
class EdgeCondUNet(nn.Module):
    """Like CondUNet but first conv accepts 2 input channels (image + edge)."""
    def __init__(self, base_ch=32, time_dim=64):
        super().__init__()
        self.class_embed = nn.Embedding(NUM_CLASSES + 1, time_dim)
        self.time_embed  = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim*4), nn.SiLU(),
            nn.Linear(time_dim*4, time_dim))
        C = base_ch
        self.enc1 = ResBlock(2, C,   time_dim)   # ← 2 input channels
        self.enc2 = ResBlock(C, C*2, time_dim)
        self.enc3 = ResBlock(C*2, C*4, time_dim)
        self.bot  = ResBlock(C*4, C*4, time_dim)
        self.dec3 = ResBlock(C*8, C*2, time_dim)
        self.dec2 = ResBlock(C*4, C,   time_dim)
        self.dec1 = ResBlock(C*2, C,   time_dim)
        self.out  = nn.Conv2d(C, 1, 1)
        self.down = nn.MaxPool2d(2)
        self.up   = nn.Upsample(scale_factor=2, mode="nearest")
    def forward(self, x, edge, t, c):
        te  = self.time_embed(t) + self.class_embed(c)
        inp = torch.cat([x, edge], dim=1)   # (B, 2, H, W)
        s1  = self.enc1(inp, te); s2 = self.enc2(self.down(s1), te)
        s3  = self.enc3(self.down(s2), te); b = self.bot(s3, te)
        d3  = self.dec3(torch.cat([self.up(b), s3], 1), te)
        d2  = self.dec2(torch.cat([self.up(d3), s2], 1), te)
        d1  = self.dec1(torch.cat([d2, s1], 1), te)
        return self.out(d1)

ctrl_model = EdgeCondUNet()
ctrl_optim = torch.optim.Adam(ctrl_model.parameters(), lr=2e-4)

ctrl_model.train()
for epoch in range(20):
    for x0, c in loader:
        t    = torch.randint(0, T_STEPS, (len(x0),))
        eps  = torch.randn_like(x0)
        xt   = sqrt_ab[t].view(-1,1,1,1)*x0 + sqrt_1mab[t].view(-1,1,1,1)*eps
        edge = sobel_edges(x0).detach()
        mask = torch.rand(len(x0)) < 0.10
        c_in = c.clone(); c_in[mask] = NULL_CLASS
        loss = F.mse_loss(ctrl_model(xt, edge, t, c_in), eps)
        ctrl_optim.zero_grad(); loss.backward(); ctrl_optim.step()
    if (epoch+1) % 5 == 0: print(f"Epoch {epoch+1}/20  loss={loss.item():.4f}")
print("Edge-conditioned U-Net trained.")

In [ ]:
def ctrl_sample(ctrl_model, target_class, source_img, n_steps=40, w=3.0):
    """
    TODO #7: Implement `ctrl_sample()`.

    Steps:
    1. Define helper function `ctrl_sample()`
    2. Compute `src`
    3. Plot results -- call `imshow()`
    4. Plot results -- call `set_ylabel()`

    Hint:
    cls = torch.full(???)
    null = torch.full(???)
    x = torch.randn_like(???)
    ts = torch.linspace(???)

    Returns: x
    """
    raise NotImplementedError("TODO: implement ctrl_sample()")

## 5 · PixelSmith v5 — Real ControlNet (optional)

```python
# Uncomment to run (downloads SD 1.5 + ControlNet Canny, ~5 GB, ~5 min CPU)
# subprocess.run([sys.executable, "-m", "pip", "install",
# "diffusers", "transformers", "accelerate", "controlnet-aux", "-q"], check=True)
#
# from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
# from controlnet_aux import CannyDetector
# from PIL import Image
# import requests, io
#
# controlnet = ControlNetModel.from_pretrained(
# "lllyasviel/sd-controlnet-canny", torch_dtype=torch.float32)
# pipe = StableDiffusionControlNetPipeline.from_pretrained(
# "runwayml/stable-diffusion-v1-5",
# controlnet=controlnet, torch_dtype=torch.float32)
# pipe = pipe.to("cpu")
#
# canny = CannyDetector()
# # Load any sketch image here
# sketch = Image.new("RGB", (512, 512), "white") # blank placeholder
# canny_img = canny(sketch)
#
# result = pipe(
# prompt="a photorealistic studio portrait, cinematic lighting",
# image=canny_img,
# num_inference_steps=20,
# guidance_scale=7.5,
# ).images[0]
# result.save("pixelsmith_v5.png")
# plt.imshow(result); plt.axis("off"); plt.show()
```

**Next:** [TextToVideo.md](../TextToVideo/TextToVideo.md) — add time.